In [ ]:
#| default_exp features.fsr

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
#| export
class FSREvaluator(FeatureEvaluator):
    """Extracts the short-to-long fragment size ratio across predefined structural metrics."""
    
    name = "FSR"
    source_file = ".FSR.parquet"
    tier = 1
    category = "fragmentation"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
                
            cols = set(df.columns)
            
            # FSR file exposes global log2 ratios
            if "short_to_long_ratio" in cols:
                extracted["global_short_to_long_ratio"] = float(df["short_to_long_ratio"].iloc[0])
            if "p_value" in cols:
                extracted["global_fsr_p_value"] = float(df["p_value"].iloc[0])

            return extracted
        except Exception as e:
            log.warning("fsr_extraction_failed", error=str(e))
            return {}


In [ ]:
#| test
evaluator = FSREvaluator()
df_test = pd.DataFrame({"short_to_long_ratio": [1.5], "p_value": [0.001]})
res = evaluator.extract(df_test)
assert res["global_short_to_long_ratio"] == 1.5
